# 03. Frame Quality Prefilter Ablation

Sources: `notebooks/03-frame-quality.md`, `src/lib/cv/frameMetrics.ts`, Pertuz et al. (2013), Farneback (2003).  
Goal: compare Laplacian/FFT/Sobel blur metrics, brightness/contrast stats, optical-flow motion, and simulate Gemini API call reduction.

Input: `data/live-frames/quality-mix/*.jpg|png|jpeg` and `data/live-frames/quality-mix/labels.csv` (`filename,label`, where label is `good` or `bad`).

In [ ]:
from pathlib import Path
import itertools
import json
import math
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
DOCS_DIR = ROOT / 'docs'
ABLATION_DIR = DOCS_DIR / 'ablation-results'
PIPELINE_DIR = DOCS_DIR / 'cv-pipeline'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

print(f'ROOT={ROOT}')
print(f'OpenCV={cv2.__version__}')


def save_placeholder_png(path, title, message='No labeled data yet'):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.58, title, ha='center', va='center', fontsize=15, weight='bold')
    ax.text(0.5, 0.42, message, ha='center', va='center', fontsize=11)
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


from sklearn.metrics import auc, precision_recall_curve, precision_recall_fscore_support, roc_curve

QUALITY_DIR = DATA_DIR / 'live-frames' / 'quality-mix'
LABEL_PATH = QUALITY_DIR / 'labels.csv'

## Data Loading
If `labels.csv` does not exist, the notebook creates empty outputs and continues safely.

In [ ]:
def load_quality_labels():
    if LABEL_PATH.exists():
        df = pd.read_csv(LABEL_PATH)
    else:
        df = pd.DataFrame(columns=['filename', 'label'])
    if 'filename' not in df.columns:
        df['filename'] = []
    if 'label' not in df.columns:
        df['label'] = []
    df['path'] = df['filename'].apply(lambda x: QUALITY_DIR / str(x) if pd.notna(x) else None)
    df['exists'] = df['path'].apply(lambda p: bool(p and p.exists()))
    df['is_good'] = df['label'].astype(str).str.lower().eq('good').astype(int)
    return df

labels = load_quality_labels()
display(labels)
print(f'label rows: {len(labels)}, existing images: {labels.exists.sum() if len(labels) else 0}')

## Quality Metric Functions
TypeScript `frameMetrics.ts` maps sharpness as `clamp01(laplacianVariance / 1600)`.

In [ ]:
def read_gray(path, max_width=1280):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None and img.shape[1] > max_width:
        scale = max_width / img.shape[1]
        img = cv2.resize(img, (max_width, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    if img is None:
        raise ValueError(path)
    return img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def laplacian_variance(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def fft_high_freq_ratio(gray, radius_ratio=0.12):
    arr = gray.astype(np.float32) / 255.0
    F = np.fft.fftshift(np.fft.fft2(arr))
    mag = np.abs(F) ** 2
    h, w = gray.shape
    yy, xx = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    radius = min(h, w) * radius_ratio
    low = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2
    total = mag.sum() + 1e-9
    return float(mag[~low].sum() / total)

def sobel_energy(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(np.sqrt(gx * gx + gy * gy)))

def brightness_stats(gray):
    mean, std = cv2.meanStdDev(gray)
    return float(mean[0][0] / 255.0), float(std[0][0] / 255.0)

def optical_flow_motion(prev_gray, gray):
    if prev_gray is None:
        return np.nan
    prev = cv2.resize(prev_gray, (gray.shape[1], gray.shape[0])) if prev_gray.shape != gray.shape else prev_gray
    flow = cv2.calcOpticalFlowFarneback(prev, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return float(np.median(mag))

def clamp01(x):
    return max(0.0, min(1.0, float(x)))

## Metric Extraction
Results are saved to `docs/ablation-results/quality-metrics.csv`.

In [ ]:
metric_rows = []
prev_gray = None
for _, row in tqdm(labels[labels['exists']].iterrows(), total=int(labels['exists'].sum()) if len(labels) else 0, desc='quality images'):
    try:
        img, gray = read_gray(row.path)
        bright, contrast = brightness_stats(gray)
        lap = laplacian_variance(gray)
        metric_rows.append({
            'filename': row.filename,
            'label': row.label,
            'is_good': int(row.is_good),
            'laplacian_variance': lap,
            'sharpness_score': clamp01(lap / 1600.0),
            'fft_high_freq_ratio': fft_high_freq_ratio(gray),
            'sobel_energy': sobel_energy(gray),
            'brightness_mean': bright,
            'brightness_std': contrast,
            'flow_median': optical_flow_motion(prev_gray, gray),
        })
        prev_gray = gray
    except Exception as exc:
        warnings.warn(f'{row.filename} failed: {exc}')

metrics = pd.DataFrame(metric_rows)
if metrics.empty:
    metrics = pd.DataFrame(columns=['filename', 'label', 'is_good', 'laplacian_variance', 'sharpness_score', 'fft_high_freq_ratio', 'sobel_energy', 'brightness_mean', 'brightness_std', 'flow_median'])
metrics.to_csv(ABLATION_DIR / 'quality-metrics.csv', index=False)
display(metrics.head())

## Laplacian vs FFT vs Sobel ROC
Figures are saved to `quality-blur-comparison.png` and the handoff-compatible `quality-roc.png`.

In [ ]:
features = ['laplacian_variance', 'fft_high_freq_ratio', 'sobel_energy']
if not metrics.empty and metrics['is_good'].nunique() == 2:
    fig, ax = plt.subplots(figsize=(7, 6))
    roc_summary = []
    for feature in features:
        y = metrics['is_good'].to_numpy()
        score = metrics[feature].fillna(metrics[feature].median()).to_numpy()
        fpr, tpr, _ = roc_curve(y, score)
        score_auc = auc(fpr, tpr)
        roc_summary.append({'feature': feature, 'auc': score_auc})
        ax.plot(fpr, tpr, label=f'{feature} AUC={score_auc:.2f}')
    ax.plot([0, 1], [0, 1], '--', color='gray')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'quality-blur-comparison.png', dpi=180)
    fig.savefig(ABLATION_DIR / 'quality-roc.png', dpi=180)
    plt.show()
    display(pd.DataFrame(roc_summary).sort_values('auc', ascending=False))
else:
    print('ROC requires both good and bad labels.')
    save_placeholder_png(ABLATION_DIR / 'quality-blur-comparison.png', 'Quality blur comparison')
    save_placeholder_png(ABLATION_DIR / 'quality-roc.png', 'Quality ROC')

## Combined Filter Grid Search
Reject rule: `sharpness_score < T1 OR brightness_mean < minB OR brightness_mean > maxB OR flow_median > T2`. Outputs are saved to `quality-threshold-grid.csv`, `quality-threshold-tuning.png`, and `quality-tradeoff.png`.

In [ ]:
grid_rows = []
if not metrics.empty and metrics['is_good'].nunique() == 2:
    y_good = metrics['is_good'].to_numpy()
    for sharp_t, min_b, max_b, flow_t in itertools.product(np.linspace(0.05, 0.5, 10), [0.08, 0.10, 0.12, 0.15], [0.85, 0.90, 0.92, 0.95], [0.5, 1.0, 2.0, np.inf]):
        rejected = (
            (metrics['sharpness_score'] < sharp_t) |
            (metrics['brightness_mean'] < min_b) |
            (metrics['brightness_mean'] > max_b) |
            (metrics['flow_median'].fillna(0) > flow_t)
        ).astype(int)
        pred_good = 1 - rejected.to_numpy()
        p, r, f1, _ = precision_recall_fscore_support(y_good, pred_good, average='binary', zero_division=0)
        api_reduction = float(rejected.mean())
        false_reject_rate = float(((rejected == 1) & (metrics['is_good'] == 1)).sum() / max(1, (metrics['is_good'] == 1).sum()))
        grid_rows.append({'minSharpness': sharp_t, 'minBrightness': min_b, 'maxBrightness': max_b, 'flowMax': flow_t, 'precision_good': p, 'recall_good': r, 'f1_good': f1, 'api_reduction': api_reduction, 'false_reject_rate': false_reject_rate})

grid = pd.DataFrame(grid_rows)
if grid.empty:
    grid = pd.DataFrame(columns=['minSharpness', 'minBrightness', 'maxBrightness', 'flowMax', 'precision_good', 'recall_good', 'f1_good', 'api_reduction', 'false_reject_rate'])
grid.to_csv(ABLATION_DIR / 'quality-threshold-grid.csv', index=False)
display(grid.sort_values(['f1_good', 'api_reduction'], ascending=False).head(10) if not grid.empty else grid)

if not grid.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(grid['api_reduction'], grid['recall_good'], c=grid['minSharpness'], cmap='viridis', s=40)
    ax.set_xlabel('API call reduction')
    ax.set_ylabel('Good frame recall')
    fig.colorbar(sc, ax=ax, label='minSharpness')
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'quality-threshold-tuning.png', dpi=180)
    fig.savefig(ABLATION_DIR / 'quality-tradeoff.png', dpi=180)
    plt.show()
else:
    print('No labeled data available for grid search.')
    save_placeholder_png(ABLATION_DIR / 'quality-threshold-tuning.png', 'Quality threshold tuning')
    save_placeholder_png(ABLATION_DIR / 'quality-tradeoff.png', 'Quality tradeoff')

## API Cost Simulation and Rejected Samples
Set `cost_per_call_usd` to the current Gemini call cost to estimate total savings.

In [ ]:
cost_per_call_usd = 0.002
if not grid.empty:
    best = grid.sort_values(['f1_good', 'api_reduction'], ascending=False).iloc[0]
    total_frames = len(metrics)
    saved_calls = int(round(total_frames * best.api_reduction))
    print(json.dumps({
        'recommended_DEFAULT_OPTIONS': {
            'minSharpness': round(float(best.minSharpness), 4),
            'minBrightness': round(float(best.minBrightness), 4),
            'maxBrightness': round(float(best.maxBrightness), 4),
        },
        'api_reduction': round(float(best.api_reduction), 4),
        'false_reject_rate': round(float(best.false_reject_rate), 4),
        'estimated_saved_calls': saved_calls,
        'estimated_saved_usd': round(saved_calls * cost_per_call_usd, 4),
    }, ensure_ascii=False, indent=2))

    rejected = metrics[
        (metrics['sharpness_score'] < best.minSharpness) |
        (metrics['brightness_mean'] < best.minBrightness) |
        (metrics['brightness_mean'] > best.maxBrightness) |
        (metrics['flow_median'].fillna(0) > best.flowMax)
    ].head(8)
    if not rejected.empty:
        fig, axes = plt.subplots(1, len(rejected), figsize=(2.5 * len(rejected), 3))
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, rejected.iterrows()):
            img = cv2.imread(str(QUALITY_DIR / row.filename))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(str(row.label))
            ax.axis('off')
        fig.tight_layout()
        fig.savefig(PIPELINE_DIR / 'quality-rejected-samples.png', dpi=180)
        plt.show()
else:
    save_placeholder_png(PIPELINE_DIR / 'quality-rejected-samples.png', 'Quality rejected samples')
    print('Fill labeled data, then run all cells.')